[![Open In Colab](https://colab.research.google.com/assets/colab-badge.png)](https://colab.research.google.com/github/lgonperez3-glitch/MLSM-labs/blob/main/Lab3/06_01_autoencoders_mtat_v5.ipynb)

# Lecture 6: Self-supervised learning and autoencoders

## MagnaTagATune — 1-second segments

This notebook trains one deeper autoencoder on MagnaTagATune (MTAT). It uses 30% of the training recordings, the full validation split, and a fixed bottleneck compression rate of 31.25:1.


In [ ]:
# Install deps

!pip install torchcodec
!pip install torch
!pip install mirdata
!pip install tqdm
!pip install scikit-learn
!pip install torchaudio
!pip install matplotlib
!pip install umap-learn

In [ ]:
import os
import json
import shutil
import subprocess
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import umap

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm, trange
from IPython.display import Audio

# Set to cuda or cpu, depending on the runtime
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Dataset preparation — MagnaTagATune

The dataset is split by original recording into training and validation subsets (80/20) before sampling. A reproducible 30% subset of the training recordings is used; the validation set remains unchanged. Each sample is one randomly selected 1-second segment from a recording.


In [ ]:
# Mount Google Drive and define the requested results folder
from google.colab import drive
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount('/content/drive')
else:
    print("Google Drive is already mounted.")

RESULTS_DIR = "/content/drive/MyDrive/MLSM/Lab 3/results_v5"
os.makedirs(RESULTS_DIR, exist_ok=True)

# MagnaTagATune files hosted by City, University of London
MTAT_ROOT = "/content/mtat"
AUDIO_DIR = os.path.join(MTAT_ROOT, "mp3")
ANNOTATIONS_PATH = os.path.join(MTAT_ROOT, "annotations_final.csv")

os.makedirs(MTAT_ROOT, exist_ok=True)

# Resume interrupted downloads and verify each file against the server's file size.
import urllib.request

def download_with_resume(url, destination):
    request = urllib.request.Request(url, method="HEAD")
    with urllib.request.urlopen(request, timeout=30) as response:
        expected_size = int(response.headers.get("Content-Length", 0))

    if os.path.exists(destination) and expected_size and os.path.getsize(destination) > expected_size:
        os.remove(destination)

    subprocess.run(["wget", "-c", "--tries=5", "--timeout=30", url, "-P", MTAT_ROOT], check=True)

    downloaded_path = os.path.join(MTAT_ROOT, os.path.basename(url))
    if not os.path.exists(downloaded_path):
        raise FileNotFoundError(f"Download did not create: {downloaded_path}")

    actual_size = os.path.getsize(downloaded_path)
    if expected_size and actual_size != expected_size:
        raise RuntimeError(
            f"Incomplete download for {os.path.basename(url)}: "
            f"{actual_size:,} bytes downloaded, expected {expected_size:,}. "
            "Run this cell again; wget will resume the download."
        )
    if actual_size == 0:
        raise RuntimeError(f"Downloaded file is empty: {downloaded_path}")

    return downloaded_path

# Check annotations on every run; wget -c skips a complete file and resumes a partial one.
download_with_resume(
    "https://mirg.city.ac.uk/datasets/magnatagatune/annotations_final.csv",
    ANNOTATIONS_PATH
)

# The archive contains 16 shards (0-9 and a-f), extracted inside AUDIO_DIR.
expected_shards = set("0123456789abcdef")
os.makedirs(AUDIO_DIR, exist_ok=True)

# Migrate shards if a previous run extracted them one directory too high.
for shard in expected_shards:
    old_location = os.path.join(MTAT_ROOT, shard)
    new_location = os.path.join(AUDIO_DIR, shard)
    if os.path.isdir(old_location) and not os.path.exists(new_location):
        shutil.move(old_location, new_location)

existing_shards = set(os.listdir(AUDIO_DIR))
audio_is_complete = expected_shards.issubset(existing_shards)

if not audio_is_complete:
    archive_parts = ["mp3.zip.001", "mp3.zip.002", "mp3.zip.003"]
    for part in archive_parts:
        download_with_resume(
            f"https://mirg.city.ac.uk/datasets/magnatagatune/{part}",
            os.path.join(MTAT_ROOT, part)
        )

    # Rebuild the joined archive from the verified parts.
    archive_path = os.path.join(MTAT_ROOT, "mp3.zip")
    with open(archive_path, "wb") as output_file:
        for part in archive_parts:
            part_path = os.path.join(MTAT_ROOT, part)
            with open(part_path, "rb") as input_file:
                shutil.copyfileobj(input_file, output_file)

    # The official archive contains folders 0-f at its root: extract them into AUDIO_DIR.
    subprocess.run(["unzip", "-q", "-o", archive_path, "-d", AUDIO_DIR], check=True)

    extracted_shards = set(os.listdir(AUDIO_DIR))
    missing_shards = sorted(expected_shards - extracted_shards)
    if missing_shards:
        raise RuntimeError(
            f"MTAT extraction is incomplete. Missing audio folders: {missing_shards}. "
            f"Top-level entries found: {sorted(extracted_shards)[:30]}"
        )

print("Audio directory:", AUDIO_DIR)
print("Annotations file:", ANNOTATIONS_PATH)
print("Audio directory exists:", os.path.isdir(AUDIO_DIR))


In [ ]:
# Read MTAT annotations
annotations = pd.read_csv(ANNOTATIONS_PATH, sep="\t")

# Exclude three empty MP3 files
invalid_files = {
    "6/norine_braun-now_and_zen-08-gently-117-146.mp3",
    "8/jacob_heringman-josquin_des_prez_lute_settings-19-gintzler__pater_noster-204-233.mp3",
    "9/american_baroque-dances_and_suites_of_rameau_and_couperin-25-le_petit_rien_xiveme_ordre_couperin-88-117.mp3",
}

annotations = annotations[
    ~annotations["mp3_path"].isin(invalid_files)
].copy()

# Identify metadata columns and tag columns
metadata_columns = [column for column in ["clip_id", "mp3_path"] if column in annotations.columns]
tag_columns = [column for column in annotations.columns if column not in metadata_columns]

# Keep rows with an available audio path and at least one positive tag
annotations = annotations.dropna(subset=["mp3_path"]).copy()
annotations[tag_columns] = annotations[tag_columns].apply(pd.to_numeric, errors="coerce").fillna(0)
annotations = annotations.loc[annotations[tag_columns].sum(axis=1) > 0].reset_index(drop=True)

# Use the most strongly annotated tag as the display label for UMAP
annotations["natural_label"] = annotations[tag_columns].idxmax(axis=1)

# Split by original recording, not by segment, to avoid data leakage
train_rows, val_rows = train_test_split(
    np.arange(len(annotations)),
    test_size=0.2,
    random_state=42,
    shuffle=True
)

train_annotations = annotations.iloc[train_rows].reset_index(drop=True)
val_annotations = annotations.iloc[val_rows].reset_index(drop=True)

# Use the same 30% training-data subsampling criterion as the Lab 4 reference.
# Keep the validation set unchanged and sample reproducibly.
full_train_size = len(train_annotations)
train_annotations = train_annotations.sample(
    n=int(full_train_size * 0.3),
    random_state=42
).reset_index(drop=True)

print(f"Tracks: {len(annotations)}")
print(f"Training tracks before subsampling: {full_train_size}")
print(f"Training tracks used (30%): {len(train_annotations)}")
print(f"Validation tracks (100%): {len(val_annotations)}")
print(f"Number of tags: {len(tag_columns)}")


In [ ]:
class MTATSegmentDataset(Dataset):
    """MTAT dataset returning one random 1-second segment per recording."""

    def __init__(
        self,
        annotations,
        audio_root,
        sample_rate=16000,
        segment_duration=1
    ):
        self.annotations = annotations.reset_index(drop=True)
        self.audio_root = audio_root
        self.sample_rate = sample_rate
        self.segment_duration = segment_duration
        self.segment_samples = sample_rate * segment_duration

    def __len__(self):
        # One sample per recording in each epoch.
        return len(self.annotations)

    def __getitem__(self, idx):
        row = self.annotations.iloc[idx]
        audio_path = os.path.join(self.audio_root, str(row["mp3_path"]))

        # Load and decode the MP3 every time a sample is requested.
        audio, sr = torchaudio.load(audio_path)

        # Convert to mono.
        if audio.shape[0] > 1:
            audio = audio.mean(dim=0, keepdim=True)

        # Resample to 16 kHz when necessary.
        if sr != self.sample_rate:
            resampler = torchaudio.transforms.Resample(sr, self.sample_rate)
            audio = resampler(audio)

        audio = audio.squeeze(0)

        # Normalize the complete recording.
        max_val = torch.abs(audio).max()
        if max_val > 0:
            audio = audio / max_val

        # Select one random 1-second segment from the first 27 seconds.
        max_start_second = max(
            0,
            int(np.ceil(len(audio) / self.sample_rate)) - self.segment_duration
        )
        max_start_second = min(max_start_second, 26)
        start_second = torch.randint(0, max_start_second + 1, (1,)).item()

        start = start_second * self.sample_rate
        end = start + self.segment_samples
        segment = audio[start:end]

        if len(segment) < self.segment_samples:
            segment = F.pad(segment, (0, self.segment_samples - len(segment)))

        return {
            "audio": segment,
            "natural_labels": row["natural_label"],
            "track_id": str(row["clip_id"]) if "clip_id" in row else str(idx)
        }


train_dataset = MTATSegmentDataset(train_annotations, AUDIO_DIR)
val_dataset = MTATSegmentDataset(val_annotations, AUDIO_DIR)

train_loader = DataLoader(
    train_dataset, batch_size=128, shuffle=True, num_workers=2,
    pin_memory=(device == "cuda")
)
val_loader = DataLoader(
    val_dataset, batch_size=128, shuffle=False, num_workers=2,
    pin_memory=(device == "cuda")
)

print(f"Training recordings: {len(train_dataset)}")
print(f"Validation recordings: {len(val_dataset)}")
print(f"Segment length: {train_dataset.segment_samples} samples ({train_dataset.segment_duration} second)")
print("Audio storage: loaded on demand; one random segment per recording; no preload or cache")


# STFT model architecture

The STFT experiments use the original three-block convolutional autoencoder architecture. The baseline is not trained; this architecture is retained only as the common model for the three perceptual-loss experiments.

In [ ]:
import torch
import torch.nn as nn

class AudioAutoencoder(nn.Module):
    """A more capable audio autoencoder featuring batch-norm, PReLu activations,
    and sharedd indices for the downsampling and upsampling layers.
    """
    def __init__(
        self,
        input_length=16000,
        bottleneck_size=64,
        kernel_size=5,
        channels=8,
        channel_mult=2,
        downsample=4,
    ):
        super().__init__()
        assert kernel_size % 2 == 1, "kernel_size should be odd for same padding"

        padding = kernel_size // 2

        # ----- Encoder -----
        self.enc_conv1 = nn.Conv1d(1, channels, kernel_size=kernel_size, padding=padding)
        self.enc_norm1 = nn.BatchNorm1d(channels)
        self.enc_act1 = nn.PReLU()
        self.pool1 = nn.MaxPool1d(2, stride=downsample, return_indices=True)

        self.enc_conv2 = nn.Conv1d(channels, channels * channel_mult, kernel_size=kernel_size, padding=padding)
        self.enc_norm2 = nn.BatchNorm1d(channels * channel_mult)
        self.enc_act2 = nn.PReLU()
        self.pool2 = nn.MaxPool1d(2, stride=downsample, return_indices=True)

        self.enc_conv3 = nn.Conv1d(channels * channel_mult, channels * channel_mult ** 2, kernel_size=kernel_size, padding=padding)
        self.enc_norm3 = nn.BatchNorm1d(channels * channel_mult ** 2)
        self.enc_act3 = nn.PReLU()
        self.pool3 = nn.MaxPool1d(2, stride=downsample, return_indices=True)

        enc_out_len = input_length // downsample ** 3 * channels * channel_mult ** 2  # 3× pooling halves length thrice
        self.enc_fc = nn.Linear(enc_out_len, bottleneck_size)

        comp_rate =  input_length / bottleneck_size
        print(f"Compression rate: {comp_rate:.2f}:1")

        # ----- Decoder -----
        self.dec_fc = nn.Linear(bottleneck_size, enc_out_len)

        self.unpool3 = nn.MaxUnpool1d(2, stride=downsample)
        self.dec_deconv3 = nn.Conv1d(channels * channel_mult ** 2, channels * channel_mult, kernel_size=kernel_size, padding=padding)
        self.dec_act3 = nn.PReLU()

        self.unpool2 = nn.MaxUnpool1d(2, stride=downsample)
        self.dec_deconv2 = nn.Conv1d(channels * channel_mult, channels, kernel_size=kernel_size, padding=padding)
        self.dec_act2 = nn.PReLU()

        self.unpool1 = nn.MaxUnpool1d(2, stride=downsample)
        self.dec_deconv1 = nn.Conv1d(channels, 1, kernel_size=kernel_size, padding=padding)
        self.dec_tanh = nn.Tanh()

    def encoder(self, x):
        sizes, indices = [], []

        x = self.enc_act1(self.enc_norm1(self.enc_conv1(x)))
        sizes.append(x.size())
        x, idx1 = self.pool1(x)
        indices.append(idx1)

        x = self.enc_act2(self.enc_norm2(self.enc_conv2(x)))
        sizes.append(x.size())
        x, idx2 = self.pool2(x)
        indices.append(idx2)

        x = self.enc_act3(self.enc_norm3(self.enc_conv3(x)))
        sizes.append(x.size())
        x, idx3 = self.pool3(x)
        indices.append(idx3)
        x = x.flatten(start_dim=1)

        x = self.enc_fc(x)
        return x, indices, sizes

    # ----- Decoder -----
    def decoder(self, z, indices, sizes):
        idx1, idx2, idx3 = indices
        size1, size2, size3 = sizes

        x = self.dec_fc(z)
        x = x.view(size3[0], size3[1], -1)

        x = self.unpool3(x, idx3, output_size=size3)
        x = self.dec_act3(self.dec_deconv3(x))

        x = self.unpool2(x, idx2, output_size=size2)
        x = self.dec_act2(self.dec_deconv2(x))

        x = self.unpool1(x, idx1, output_size=size1)
        x = self.dec_tanh(self.dec_deconv1(x))

        return x

    # ----- Forward -----
    def forward(self, x):
        z, indices, sizes = self.encoder(x)
        x_r = self.decoder(z, indices, sizes)
        return x_r, z

In [ ]:
# Keep the original compression rate: 16,000 / 512 = 31.25
target_length = 16000
bottleneck_size = 512
downsample = 2
num_epochs = 20

COMPRESSION_RATE = target_length / bottleneck_size
assert abs(COMPRESSION_RATE - 31.25) < 1e-8
print(f"Compression rate: {COMPRESSION_RATE:.2f}:1")

# Shared training/evaluation routine used by the improved and STFT experiments
def evaluate_model(model, loader, stft_config=None):
    model.eval()
    mse_values = []
    stft_values = []

    with torch.no_grad():
        for batch in loader:
            x = batch["audio"].unsqueeze(1).to(device)
            x_reconstr, _ = model(x)
            mse_values.append(F.mse_loss(x_reconstr, x).item())

            if stft_config is not None:
                stft_values.append(advanced_perceptual_loss(x, x_reconstr, **stft_config).item())

    return float(np.mean(mse_values)), (float(np.mean(stft_values)) if stft_values else np.nan)


def train_model(model, model_name, loss_mode="mse", stft_config=None):
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    history = {
        "train_mse": [],
        "val_mse": [],
        "train_stft": [],
        "val_stft": [],
        "train_objective": [],
    }

    pbar = trange(num_epochs, desc=f"Training {model_name}", unit="epoch")
    for epoch in pbar:
        model.train()
        train_mse = []
        train_stft = []
        train_objective = []

        for batch in train_loader:
            x = batch["audio"].unsqueeze(1).to(device)
            optimizer.zero_grad()

            x_reconstr, _ = model(x)
            mse_loss = criterion(x_reconstr, x)

            if loss_mode == "mse":
                loss = mse_loss
                stft_loss = torch.tensor(float("nan"), device=device)
            else:
                stft_loss = advanced_perceptual_loss(x, x_reconstr, **stft_config)
                loss = stft_config["mse_weight"] * mse_loss + stft_config["stft_weight"] * stft_loss

            loss.backward()
            optimizer.step()

            train_mse.append(mse_loss.item())
            train_objective.append(loss.item())
            if loss_mode != "mse":
                train_stft.append(stft_loss.item())

        val_mse, val_stft = evaluate_model(
            model, val_loader,
            stft_config=stft_config if loss_mode != "mse" else None
        )

        history["train_mse"].append(float(np.mean(train_mse)))
        history["val_mse"].append(val_mse)
        history["train_objective"].append(float(np.mean(train_objective)))
        history["train_stft"].append(float(np.mean(train_stft)) if train_stft else np.nan)
        history["val_stft"].append(val_stft)

        pbar.set_postfix({
            "train MSE": f"{history['train_mse'][-1]:.6f}",
            "val MSE": f"{val_mse:.6f}",
            "val STFT": f"{val_stft:.5f}" if not np.isnan(val_stft) else "n/a"
        })

    torch.save(model.state_dict(), os.path.join(RESULTS_DIR, f"{model_name}.pt"))
    with open(os.path.join(RESULTS_DIR, f"{model_name}_history.json"), "w") as file:
        json.dump(history, file, indent=2)

    return model, history


# 1. Improved autoencoder — additional convolutional layers

The improved model adds two convolutional/pooling blocks to the encoder and mirrored decoder. It is trained first, using MSE reconstruction loss while keeping the bottleneck compression rate fixed at 31.25:1.

In [ ]:
class AudioAutoencoderImproved(nn.Module):
    """Deeper version of the original autoencoder with mirrored encoder/decoder blocks."""

    def __init__(
        self,
        input_length=16000,
        bottleneck_size=512,
        kernel_size=5,
        channels=8,
        channel_mult=2,
        downsample=2,
        extra_layers=1,
    ):
        super().__init__()
        assert kernel_size % 2 == 1, "kernel_size should be odd for same padding"

        self.num_layers = 3 + extra_layers
        self.downsample = downsample
        padding = kernel_size // 2

        self.encoder_convs = nn.ModuleList()
        self.encoder_norms = nn.ModuleList()
        self.encoder_acts = nn.ModuleList()
        self.pools = nn.ModuleList()

        self.channel_sizes = [channels * (channel_mult ** i) for i in range(self.num_layers)]

        in_channels = 1
        for out_channels in self.channel_sizes:
            self.encoder_convs.append(
                nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, padding=padding)
            )
            self.encoder_norms.append(nn.BatchNorm1d(out_channels))
            self.encoder_acts.append(nn.PReLU())
            self.pools.append(nn.MaxPool1d(2, stride=downsample, return_indices=True))
            in_channels = out_channels

        enc_out_len = (input_length // downsample ** self.num_layers) * self.channel_sizes[-1]
        self.enc_fc = nn.Linear(enc_out_len, bottleneck_size)
        self.dec_fc = nn.Linear(bottleneck_size, enc_out_len)

        self.unpools = nn.ModuleList()
        self.decoder_convs = nn.ModuleList()
        self.decoder_acts = nn.ModuleList()

        for layer_idx in reversed(range(self.num_layers)):
            in_ch = self.channel_sizes[layer_idx]
            out_ch = self.channel_sizes[layer_idx - 1] if layer_idx > 0 else 1
            self.unpools.append(nn.MaxUnpool1d(2, stride=downsample))
            self.decoder_convs.append(
                nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, padding=padding)
            )
            self.decoder_acts.append(nn.PReLU() if layer_idx > 0 else nn.Tanh())

        comp_rate = input_length / bottleneck_size
        print(f"Improved compression rate: {comp_rate:.2f}:1 | Extra layers: {extra_layers}")

    def encoder(self, x):
        sizes, indices = [], []

        for conv, norm, act, pool in zip(
            self.encoder_convs, self.encoder_norms, self.encoder_acts, self.pools
        ):
            x = act(norm(conv(x)))
            sizes.append(x.size())
            x, idx = pool(x)
            indices.append(idx)

        x = x.flatten(start_dim=1)
        z = self.enc_fc(x)
        return z, indices, sizes

    def decoder(self, z, indices, sizes):
        x = self.dec_fc(z)
        x = x.view(sizes[-1][0], sizes[-1][1], -1)

        for idx, size, unpool, conv, act in zip(
            reversed(indices), reversed(sizes), self.unpools,
            self.decoder_convs, self.decoder_acts
        ):
            x = unpool(x, idx, output_size=size)
            x = act(conv(x))

        return x

    def forward(self, x):
        z, indices, sizes = self.encoder(x)
        x_reconstructed = self.decoder(z, indices, sizes)
        return x_reconstructed, z

In [ ]:
improved_results = {}

# Train the deeper architecture with two additional encoder/decoder blocks.
extra_layers = 2
model_name = f"improved_{extra_layers}_extra_layers"

improved_model = AudioAutoencoderImproved(
    input_length=target_length,
    bottleneck_size=bottleneck_size,
    downsample=downsample,
    extra_layers=extra_layers
)

improved_model, improved_history = train_model(
    improved_model,
    model_name,
    loss_mode="mse"
)

improved_results[model_name] = {
    "model": improved_model,
    "history": improved_history
}

# 3. Improve reconstruction with perceptual losses — STFT models

The original multi-scale STFT magnitude loss is retained as the first configuration. Two additional configurations extend it with spectral convergence, log-magnitude distance and a weighted multi-resolution objective. Each STFT experiment is trained independently and is also evaluated in MSE, so its reconstruction quality can be compared fairly with the baseline and improved models.

In [ ]:
def multi_scale_stft_loss(x, y, scales=[512, 1024, 2048], hop_ratio=0.25):
    """Simple implementation of the multi-scale STFT loss.
    https://arxiv.org/pdf/1910.11480
    """
    loss = 0
    for n_fft in scales:
        hop = int(n_fft * hop_ratio)
        window = torch.hann_window(n_fft, device=x.device)
        X = torch.stft(x.squeeze(), n_fft=n_fft, hop_length=hop, window=window, return_complex=True)
        Y = torch.stft(y.squeeze(), n_fft=n_fft, hop_length=hop, window=window, return_complex=True)
        loss += torch.mean(torch.abs(X.abs() - Y.abs()))
    return loss / len(scales)

In [ ]:
def advanced_perceptual_loss(
    x,
    y,
    scales=(512, 1024, 2048),
    hop_ratio=0.25,
    spectral_convergence_weight=0.0,
    log_magnitude_weight=0.0,
    magnitude_weight=1.0,
):
    """Multi-resolution STFT loss with optional spectral convergence and log-magnitude terms."""
    magnitude_losses = []
    convergence_losses = []
    log_losses = []

    x = x.squeeze(1)
    y = y.squeeze(1)

    for n_fft in scales:
        hop = int(n_fft * hop_ratio)
        window = torch.hann_window(n_fft, device=x.device)

        X = torch.stft(
            x, n_fft=n_fft, hop_length=hop, window=window,
            return_complex=True
        )
        Y = torch.stft(
            y, n_fft=n_fft, hop_length=hop, window=window,
            return_complex=True
        )

        X_mag = X.abs()
        Y_mag = Y.abs()
        magnitude_losses.append(F.l1_loss(X_mag, Y_mag))

        eps = 1e-7
        numerator = torch.linalg.vector_norm((X_mag - Y_mag).flatten(start_dim=1), dim=1)
        denominator = torch.linalg.vector_norm(X_mag.flatten(start_dim=1), dim=1).clamp_min(eps)
        convergence_losses.append((numerator / denominator).mean())

        log_losses.append(
            F.l1_loss(torch.log(X_mag + eps), torch.log(Y_mag + eps))
        )

    magnitude_loss = torch.stack(magnitude_losses).mean()
    convergence_loss = torch.stack(convergence_losses).mean()
    log_loss = torch.stack(log_losses).mean()

    return (
        magnitude_weight * magnitude_loss
        + spectral_convergence_weight * convergence_loss
        + log_magnitude_weight * log_loss
    )

In [ ]:
# STFT configurations: original multi-scale magnitude loss, then two enhanced perceptual objectives
stft_configs = {
    "stft_original": {
        "scales": (512, 1024, 2048),
        "hop_ratio": 0.25,
        "spectral_convergence_weight": 0.0,
        "log_magnitude_weight": 0.0,
        "magnitude_weight": 1.0,
        "mse_weight": 0.5,
        "stft_weight": 1.0
    },
    "stft_sc_log": {
        "scales": (512, 1024, 2048),
        "hop_ratio": 0.25,
        "spectral_convergence_weight": 1.0,
        "log_magnitude_weight": 1.0,
        "magnitude_weight": 1.0,
        "mse_weight": 0.5,
        "stft_weight": 1.0
    },
    "stft_weighted_multires": {
        "scales": (256, 512, 1024, 2048),
        "hop_ratio": 0.25,
        "spectral_convergence_weight": 1.0,
        "log_magnitude_weight": 2.0,
        "magnitude_weight": 0.5,
        "mse_weight": 0.5,
        "stft_weight": 1.0
    }
}

stft_results = {}

for model_name, stft_config in stft_configs.items():
    stft_model = AudioAutoencoder(
        input_length=target_length,
        bottleneck_size=bottleneck_size,
        downsample=downsample
    )

    stft_model, stft_history = train_model(
        stft_model,
        model_name,
        loss_mode="stft",
        stft_config=stft_config
    )

    stft_results[model_name] = {
        "model": stft_model,
        "history": stft_history,
        "config": stft_config
    }

# 3. Compare validation MSE during training

The following figure compares validation MSE for the improved autoencoder and all STFT perceptual-loss configurations. The baseline is not trained or included.

In [ ]:
all_results = {}
all_results.update(improved_results)
all_results.update(stft_results)

plt.figure(figsize=(12, 7))

for model_name, result in all_results.items():
    plt.plot(
        range(1, len(result["history"]["val_mse"]) + 1),
        result["history"]["val_mse"],
        marker="o",
        markersize=3,
        label=model_name
    )

plt.title("Validation MSE during training — improved and STFT models")
plt.xlabel("Epoch")
plt.ylabel("Validation MSE")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "validation_mse_comparison.png"), dpi=200)
plt.show()

# 5. STFT loss versus MSE

This plot tracks the validation STFT perceptual loss against validation MSE across epochs for each STFT configuration. It helps show whether perceptual improvements are accompanied by changes in waveform reconstruction error.

In [ ]:
plt.figure(figsize=(10, 7))

for model_name, result in stft_results.items():
    history = result["history"]
    plt.plot(
        history["val_mse"],
        history["val_stft"],
        marker="o",
        label=model_name
    )

plt.title("Validation STFT perceptual loss versus validation MSE")
plt.xlabel("Validation MSE")
plt.ylabel("Validation STFT loss")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "stft_vs_mse.png"), dpi=200)
plt.show()

# 5. UMAP projections of the latent representations

A separate UMAP is generated for each trained configuration (the improved model and all STFT models) using validation embeddings.

In [ ]:
def plot_umap(model, model_name, max_points=5000):
    model.eval()
    embeddings = []
    natural_labels = []
    collected = 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Extracting embeddings: {model_name}"):
            x = batch["audio"].unsqueeze(1).to(device)
            _, z = model(x)

            embeddings.append(z.cpu().numpy())
            natural_labels.extend(batch["natural_labels"])
            collected += len(z)

            if collected >= max_points:
                break

    embeddings = np.concatenate(embeddings, axis=0)[:max_points]
    natural_labels = np.asarray(natural_labels)[:max_points]

    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=15,
        min_dist=0.1,
        random_state=42
    )
    umap_embeddings = reducer.fit_transform(embeddings)

    unique_labels, label_indices = np.unique(natural_labels, return_inverse=True)

    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(
        umap_embeddings[:, 0],
        umap_embeddings[:, 1],
        c=label_indices,
        cmap="tab20",
        s=6,
        alpha=0.75
    )

    handles, _ = scatter.legend_elements(num=min(len(unique_labels), 20))
    legend_labels = unique_labels[:len(handles)]
    plt.legend(handles, legend_labels, title="Most common MTAT tag", loc="best", fontsize=7)

    plt.title(f"UMAP of Audio Embeddings — {model_name}")
    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f"umap_{model_name}.png"), dpi=200)
    plt.show()


for model_name, result in all_results.items():
    plot_umap(result["model"], model_name)

# 6. Final metrics

In [ ]:
# Save a compact summary of the final validation metrics and all experiment settings
summary = []

for model_name, result in all_results.items():
    history = result["history"]
    row = {
        "model": model_name,
        "final_train_mse": history["train_mse"][-1],
        "final_validation_mse": history["val_mse"][-1],
        "compression_rate": COMPRESSION_RATE,
        "segment_duration_seconds": 1,
        "sample_rate": 16000
    }
    if model_name in stft_results:
        row["final_validation_stft"] = history["val_stft"][-1]
        row["stft_config"] = json.dumps(stft_results[model_name]["config"])
    else:
        row["final_validation_stft"] = np.nan
        row["stft_config"] = ""
    summary.append(row)

summary_df = pd.DataFrame(summary)
summary_df.to_csv(os.path.join(RESULTS_DIR, "training_summary.csv"), index=False)
summary_df

# 7. Listen to the original audio and reconstructions

All trained models are evaluated on the same five consecutive 1-second segments from one validation recording. Each segment is reconstructed independently, and the five outputs are concatenated for comparison.

In [ ]:
from IPython.display import Audio, display

# Select one validation recording and apply the same audio preprocessing.
reference_row = val_annotations.iloc[0]
reference_path = os.path.join(AUDIO_DIR, str(reference_row["mp3_path"]))
audio, sr = torchaudio.load(reference_path)

if audio.shape[0] > 1:
    audio = audio.mean(dim=0, keepdim=True)
if sr != 16000:
    audio = torchaudio.transforms.Resample(sr, 16000)(audio)

audio = audio.squeeze(0)
peak = audio.abs().max()
if peak > 0:
    audio = audio / peak

# Select five consecutive seconds within the first 27 seconds.
segment_samples = 16000
max_start = min(max(0, len(audio) // segment_samples - 5), 22)
start_second = int(torch.randint(0, max_start + 1, (1,)).item())
start = start_second * segment_samples
original = audio[start:start + 5 * segment_samples]

if len(original) < 5 * segment_samples:
    original = F.pad(original, (0, 5 * segment_samples - len(original)))

original_segments = original.split(segment_samples)
print("Reference track:", reference_row["clip_id"])
print("MTAT tag:", reference_row["natural_label"])
print("Start time:", start_second, "seconds")
print("\nORIGINAL AUDIO — 5 SECONDS")
display(Audio(original.cpu().numpy(), rate=16000))

# Reconstruct the same five segments with the improved and STFT-trained models.
for model_name, result in all_results.items():
    model = result["model"]
    model.eval()
    reconstructed_segments = []

    with torch.no_grad():
        for segment in original_segments:
            model_input = segment.unsqueeze(0).unsqueeze(0).to(device)
            reconstruction, _ = model(model_input)
            reconstructed_segments.append(reconstruction.squeeze().cpu())

    reconstructed_audio = torch.cat(reconstructed_segments)

    print(f"\nRECONSTRUCTION — {model_name} — 5 SECONDS")
    display(Audio(reconstructed_audio.numpy(), rate=16000))

    torchaudio.save(
        os.path.join(RESULTS_DIR, f"audio_reference_5segments_{model_name}.wav"),
        reconstructed_audio.unsqueeze(0),
        sample_rate=16000
    )

torchaudio.save(
    os.path.join(RESULTS_DIR, "audio_reference_5segments_original.wav"),
    original.cpu().unsqueeze(0),
    sample_rate=16000
)

# 8. Report

**Data Loading and Preprocessing**

The MagnaTagATune (MTAT) dataset is processed using an on-demand loading strategy. Each recording is decoded only when a sample is requested by the DataLoader. The recordings are split at track level into training and validation subsets using an 80/20 split and a fixed random seed. A reproducible 30% random subset of the training recordings is retained, while the full validation subset is used. Each audio recording is converted to mono, resampled to 16 kHz when necessary, normalized, and cropped to one randomly selected second from the first 27 seconds. The selected segment may vary between epochs.

The DataLoaders use a batch size of 128 and two worker processes. No audio preloading or caching is used.

**Improved Autoencoder**

The first trained model is the improved autoencoder with two additional encoder/decoder blocks. The bottleneck size is 512 for 16,000 input samples, preserving a compression rate of 31.25:1. It is trained for 20 epochs using Adam (learning rate 0.001) and mean squared error (MSE) reconstruction loss.

**STFT Perceptual-Loss Experiments**

The three STFT experiments are retained and trained independently after the improved model. They use the original three-block convolutional autoencoder architecture, without training a separate baseline model. The first configuration uses multi-scale STFT magnitude loss. The second adds spectral convergence and log-magnitude terms. The third uses a weighted multi-resolution configuration with four FFT scales. Each STFT model combines its perceptual loss with MSE using the weights specified in its configuration. All models are evaluated using validation MSE and validation STFT loss where applicable.